运行约定：以本 Notebook 所在目录作为工作目录，从第一单元顺序运行。首个代码单元仅建立数据与输出目录。历史输出保留，不代表当前环境已验证。


In [ ]:
from pathlib import Path
Path('../../data').mkdir(parents=True, exist_ok=True)
Path('../../outputs/pytorch').mkdir(parents=True, exist_ok=True)


# 一、自然语言处理的常见任务

> **本 Notebook 为 PyTorch 版本**，与原 PaddlePaddle 版本的理论内容保持一致，供使用 PyTorch 框架学习的读者参考。

自然语言处理是非常复杂的领域，是人工智能中最为困难的问题之一，常见的任务如 **图9** 所示：

<center><img src="../assets/01-nlp-overview-1.png" width="600" ></center>
<center><br>图9：自然语言处理常见任务</br></center>
<br></br>

- **词和短语级任务**：包括切词、词性标注、命名实体识别（如"苹果很好吃"和"苹果很伟大"中的"苹果"，哪个是苹果公司？）、同义词计算（如"好吃"的同义词是什么？）等以词为研究对象的任务。
- **句子和段落级任务**：包括文本倾向性分析（如客户说："你们公司的产品真好用！"是在夸赞还是在讽刺？）、文本相似度计算（如"我坐高铁去广州"和"我坐火车去广州"是一个意思吗？）等以句子为研究对象的任务。
- **对话和篇章级任务**：包括机器阅读理解（如使用医药说明书回答患者的咨询问题）、对话系统（如打造一个24小时在线的AI话务员）等复杂的自然语言处理系统等。
- **自然语言生成**：如机器翻译（如"我爱飞桨"的英文是什么？）、机器写作（以AI为题目写一首诗）等自然语言生成任务。

# 二、自然语言处理技术面临的挑战

如何让机器像人一样，能够准确理解和使用自然语言？这是当前自然语言处理领域面临的最大挑战。为了解决这一问题，我们需要从语言学和计算两个角度思考。

## 2.1 语言学角度

自然语言数量多、形态各异，理解自然语言对人来说本身也是一件复杂的事情，如同义词、情感倾向、歧义性、长文本处理、语言惯性表达等。通过如下几个例子，我们一同感受一下。

### 2.1.1 同义词问题

请问下列词语是否为同义词？（题目来源：四川话和东北话6级模拟考试）
   
   > 瓜兮兮 和 铁憨憨
   >
   > 嘎嘎 和 肉（you）
   >
   > 磕搀 和 难看
   >
   > 吭呲瘪肚 和 速度慢 
  
### 2.1.2 情感倾向问题

请问如何正确理解下面两个场景？
   
   场景一：女朋友生气了，男朋友电话道歉。
   
   > 女生：就算你买包我也不会原谅你！
   >
   > 男生：宝贝，放心，我不买，你别生气了。
   >
   > 问：女生会不会生气？
   
   场景二：甲和乙是同宿舍的室友，他们之间的对话。
   >
   > 甲：钥匙好像没了，你把锁别别。
   >
   > 乙：到底没没没？
   >
   > 甲：我也不道没没没。
   >
   > 乙：要没没你让我别，别别了，别秃鲁了咋整？
   >
   > 问：到底别不别？
   
### 2.1.3 歧义性问题

请问如何理解下面三句话？

   > 一行行行行行，一行不行行行不行。
   >
   > 来到杨过曾经生活过的地方，小龙女说："我也想过过过儿过过的生活"。
   >
   > 来到儿子等校车的地方，邓超对孙俪说："我也想等等等等等过的那辆车"。

相信大多数人都需要花点脑筋去理解上面的句子，在不同的上下文中，相同的单词可以具有不同的含义，这种问题我们称之为歧义性问题。
### 2.1.4 对话/篇章等长文本处理问题

在处理长文本（如一篇新闻报道，一段多人对话，甚至于一篇长篇小说）时，需要经常处理各种省略、指代、话题转折和切换等语言学现象，给机器理解自然语言带来了挑战，如 **图7** 所示。
   
<center><img src="../assets/01-nlp-overview-2.png" width="800" ></center>
<center><br>图7：多轮对话中的指代和省略</br></center>
<br></br>

### 2.1.5 探索自然语言理解的本质问题

   > 研表究明，汉字的顺序并不定一能影阅响读，比如当你看完这句话后，才发这现里的字全是都乱的。

上面这句话从语法角度来说完全是错的，但是对大部分人来说完全不影响理解，甚至很多人都不会意识到这句话的语法是错的。

## 2.2 计算角度

自然语言技术的发展除了受语言学的制约外，在计算角度也天然存在局限。顾名思义，计算机是计算的机器，现有的计算机都以浮点数为输入和输出，擅长执行加减乘除类计算。自然语言本身并不是浮点数，计算机为了能存储和显示自然语言，需要把自然语言中的字符转换为一个固定长度（或者变长）的二进制编码，如 **图** 所示。
<center><img src="../assets/01-nlp-overview-3.png" width="600" ></center>
<center><br>图：计算机计算自然语言流程</br></center>
<br></br>

由于这个编码本身不是数字，对这个编码的计算往往不具备数学和物理含义。例如：把"法国"和"首都"放在一起，大多数人首先联想到的内容是"巴黎"。但是如果我们使用"法国"和"首都"的UTF-8编码去做加减乘除等运算，是无法轻易获取到"巴黎"的UTF-8编码，甚至无法获得一个有效的UTF-8编码。因此，如何让计算机可以有效地计算自然语言，是计算机科学家和工程师面临的巨大挑战。

此外，目前也有研究人员正在关注自然语言处理方法中的社会问题：包括自然语言处理模型中的偏见和歧视、大规模计算对环境和气候带来的影响、传统工作被取代后，人的失业和再就业问题等。